In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


In [ ]:

text = """twinkle twinkle little star
how I wonder what you are
up above the world so high
like a diamond in the sky"""


text = text.lower()
print(text)


In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Total unique words:", total_words)


input_sequences = []

for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("Total training sequences:", len(input_sequences))


 Tokenizer() creates an empty tokenizer.
fit_on_texts() builds a word-index dictionary.
Example:
If text has "twinkle twinkle little", the tokenizer might assign:
{"twinkle": 1, "little": 2, ...}

 Counts total unique words and adds 1 because indexing starts at 1 (0 is reserved for padding).
If there are 10 words, total_words = 11.

 Splits the poem into lines (split("\n"))
texts_to_sequences converts each word into a number based on the tokenizer dictionary.

n_gram_sequence creates input patterns:

For "twinkle twinkle little"
token list: [1, 1, 2]
then it makes:

[1, 1]

[1, 1, 2]


This gives training examples like:

Input: ["twinkle"] → Output: "twinkle"

Input: ["twinkle", "twinkle"] → Output: "little"

In [ ]:

max_seq_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = to_categorical(y, num_classes=total_words)


Finds the length of the longest n-gram.

Needed to ensure all sequences are the same size when training.
pad_sequences: Adds zeros in front of shorter sequences to match the longest one.
If max length is 5:

[1, 2] becomes [0, 0, 0, 1, 2]
 Splits each sequence into.

---



X: all words except last (input)

y: the last word (label)

For [0, 0, 1, 1, 2]

X = [0, 0, 1, 1]

y = 2

Converts y (which is a number) into one-hot encoded format.
If there are 5 words in the vocab:

2 → [0, 0, 1, 0, 0]

This is required for classification.

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=total_words, output_dim=10, input_length=max_seq_len - 1))
model.add(SimpleRNN(64))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])



Begins building a model layer by layer.
 The Embedding layer:

Converts each word into a 10-dimensional vector

Learns word meaning from the data

 A simple Recurrent Neural Network layer with 64 memory units:


Final output layer:

Each neuron represents a possible next word.

softmax: gives probabilities for each word.
Compile means setting up the model for training:

loss: measures error

optimizer: improves weights (adam is a smart gradient descent optimizer)

metrics: what we use to evaluate performance (here, accuracy)

Can remember context
from earlier words (e.g., remembers "twinkle" when seeing "little")

Trains the model for 200 full passes (epochs) over the dataset.verbose=1: shows training progress bar and accuracy after each epoch.

 Function to generate next_words words based on seed_text.
Step-by-step:

Convert seed to tokens.

Pad it to match training input length.

Predict next word (argmax finds most probable index).

Convert number back to word.

Add word to the input and repeat.

Gives outputs like:

Input: "twinkle twinkle"
→ Output: "twinkle twinkle little star"

Input: "how i"
→ Output: "how i wonder what"


In [ ]:
history = model.fit(X, y, epochs=200, verbose=1)


In [ ]:
def predict_next_word(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        output_word = tokenizer.index_word.get(predicted[0], '')
        seed_text += " " + output_word
    return seed_text


In [ ]:
print(predict_next_word("twinkle twinkle",1))
print(predict_next_word("how i", 1))
